## Optuna

Key Terminologies in Optuna:

1. **Study**: A study is an optimization task that consists of multiple trials. It is the main object in Optuna that manages the optimization process.
2. **Trial**: A trial is a single execution of the objective function with a specific set of hyperparameters. Each trial evaluates the performance of the model with those hyperparameters.
3. **Objective Function**: The objective function is the function that you want to optimize. It takes hyperparameters as input and returns a value that indicates the performance of the model.
4. **Sampler**: A sampler is responsible for suggesting hyperparameter values for each trial. Optuna provides various samplers, such as TPE (Tree-structured Parzen Estimator) and Random Sampler. (Acquisition Function + Surrogate Function)
5. **Pruner**: A pruner is responsible for stopping unpromising trials early to save computational resources. Optuna provides various pruners, such as Median Pruner and Successive Halving Pruner.
6. **Hyperparameter Space**: The hyperparameter space defines the range of values that each hyperparameter can take. It can include continuous, discrete, and categorical parameters.
7. **Trial Object**: The trial object is an instance of a trial that contains information about the hyperparameters, the objective function value, and other metadata related to the trial.
8. **Distributions**: Distributions define how hyperparameters are sampled. Optuna supports various distributions, such as uniform, log-uniform, and categorical distributions.
9. **Optimization Direction**: The optimization direction specifies whether the objective function should be minimized or maximized. Optuna allows you to set the optimization direction when creating a study.
10 **Search Algorithm**: The search algorithm defines the strategy used to explore the hyperparameter space. Optuna provides different search algorithms, such as TPE, Random Search, and Grid Search.

In [1]:
## Will try Optuna on The diabetes dataset of SKLearn
import optuna
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [3]:
df = pd.read_csv(url, names = columns)

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
## Splitting data into X, Y

X = df.drop(columns= ["Outcome"])
Y = df["Outcome"]

In [6]:
# finding columns with zero values in them
zero_column_list = []

for column in X.columns:
    if (df[column] == 0).sum() > 0:
        zero_column_list.append(column)


In [7]:
zero_column_list

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

In [8]:
# Replacing zero values with nan values

X[zero_column_list] = X[zero_column_list].replace(0,np.nan, )

In [9]:
X.fillna(X.mean(), inplace = True)

In [10]:
(X==0).sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size= 0.25, random_state= 42)

In [12]:
## Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
## Define the objective function to be optimised

def objective_function(trial):
    ## defining the hyperparameter space
    n_estimator = trial.suggest_int("n_estimators", 50,200)
    max_depth = trial.suggest_int("max_depth", 3,20)
    min_sample_split = trial.suggest_int("min_samples_split", 2,10)

    ## Create the model to be trained

    model = RandomForestClassifier(
        n_estimators= n_estimator,
        max_depth= max_depth, min_samples_split= min_sample_split,
        random_state= 42
    )


    ## Perform a 3 fold cross validation and calculate accuracy

    score = cross_val_score(model, X_train, Y_train, cv= 3, scoring= "accuracy").mean()

    return score # Return the accuracy score for optuna to maximise

In [14]:
## Create a study in optuna to maximize the accuracy of the objective function

study = optuna.create_study(direction= "maximize") # We aim to maximize accuracy

study.optimize(objective_function, n_trials= 50)

[I 2026-10-17 16:03:20,624] A new study created in memory with name: no-name-f07bce0e-05de-4b21-b202-4afb06bce88f
[I 2026-10-17 16:03:20,768] Trial 0 finished with value: 0.748263888888889 and parameters: {'n_estimators': 77, 'max_depth': 16, 'min_samples_split': 2}. Best is trial 0 with value: 0.748263888888889.
[I 2026-10-17 16:03:20,912] Trial 1 finished with value: 0.75 and parameters: {'n_estimators': 115, 'max_depth': 3, 'min_samples_split': 3}. Best is trial 1 with value: 0.75.
[I 2026-10-17 16:03:21,148] Trial 2 finished with value: 0.7638888888888888 and parameters: {'n_estimators': 155, 'max_depth': 10, 'min_samples_split': 9}. Best is trial 2 with value: 0.7638888888888888.
[I 2026-10-17 16:03:21,384] Trial 3 finished with value: 0.7690972222222222 and parameters: {'n_estimators': 160, 'max_depth': 9, 'min_samples_split': 8}. Best is trial 3 with value: 0.7690972222222222.
[I 2026-10-17 16:03:21,626] Trial 4 finished with value: 0.7465277777777777 and parameters: {'n_estimat

In [15]:
#  print the best accuracy from the trials and the parameters which gave the best accuracy

print(f"The best accuracy is {study.best_trial.value}")
print(f"The parameters for best accuracy are {study.best_trial.params}")

The best accuracy is 0.7760416666666666
The parameters for best accuracy are {'n_estimators': 116, 'max_depth': 7, 'min_samples_split': 6}


In [16]:
print(f"The parameters for best accuracy are {study.best_params}")

The parameters for best accuracy are {'n_estimators': 116, 'max_depth': 7, 'min_samples_split': 6}


In [17]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_params, random_state = 42)

best_model.fit(X_train, Y_train)

y_pred = best_model.predict(X_test)

# Calculate accuracy

score = accuracy_score(Y_test, y_pred)

print(f"the besta ccuracy score for test data is {score}")

the besta ccuracy score for test data is 0.734375


## Samplers in Optuna

When we dont specify a sampler, Optuna uses the default sampler, which is the TPE (Tree-structured Parzen Estimator) sampler. The TPE sampler is a Bayesian optimization algorithm that models the objective function using a probabilistic model and selects hyperparameters based on this model.

Other kind of samplers available in Optuna are:
- **Random Sampler**: This sampler randomly samples hyperparameter values from the defined hyperparameter space. It is useful for simple optimization tasks or when you want to explore the hyperparameter space without any prior knowledge.
- **Grid Sampler**: This sampler performs a grid search over the hyperparameter space.
-**cmaes**: The CMA-ES (Covariance Matrix Adaptation Evolution Strategy) sampler is an evolutionary algorithm that adapts the covariance matrix of the search distribution based on the performance of previous trials. It is suitable for continuous optimization problems and can handle complex, non-linear objective functions.


In [18]:
def objective_function(trial):
    ## defining the hyperparameter space
    n_estimator = trial.suggest_int("n_estimators", 50,200)
    max_depth = trial.suggest_int("max_depth", 3,20)
    min_sample_split = trial.suggest_int("min_samples_split", 2,10)

    ## Create the model to be trained

    model = RandomForestClassifier(
        n_estimators= n_estimator,
        max_depth= max_depth, min_samples_split= min_sample_split,
        random_state= 42
    )


    ## Perform a 3 fold cross validation and calculate accuracy

    score = cross_val_score(model, X_train, Y_train, cv= 3, scoring= "accuracy").mean()

    return score # Return the accuracy score for optuna to maximise

In [19]:
study = optuna.create_study(direction= "maximize", sampler= optuna.samplers.RandomSampler() )

study.optimize(objective_function, n_trials = 50)

[I 2026-10-17 16:03:27,658] A new study created in memory with name: no-name-596de9ed-4477-40e5-bc12-6aaf085cfaf0
[I 2026-10-17 16:03:27,825] Trial 0 finished with value: 0.7621527777777777 and parameters: {'n_estimators': 104, 'max_depth': 19, 'min_samples_split': 6}. Best is trial 0 with value: 0.7621527777777777.
[I 2026-10-17 16:03:28,064] Trial 1 finished with value: 0.7638888888888888 and parameters: {'n_estimators': 160, 'max_depth': 8, 'min_samples_split': 2}. Best is trial 1 with value: 0.7638888888888888.
[I 2026-10-17 16:03:28,290] Trial 2 finished with value: 0.7638888888888888 and parameters: {'n_estimators': 150, 'max_depth': 8, 'min_samples_split': 2}. Best is trial 1 with value: 0.7638888888888888.
[I 2026-10-17 16:03:28,466] Trial 3 finished with value: 0.765625 and parameters: {'n_estimators': 123, 'max_depth': 6, 'min_samples_split': 4}. Best is trial 3 with value: 0.765625.
[I 2026-10-17 16:03:28,582] Trial 4 finished with value: 0.7586805555555555 and parameters: {

In [20]:
study.best_value

0.7760416666666666

In [21]:
study.best_params

{'n_estimators': 88, 'max_depth': 8, 'min_samples_split': 2}

### GridSampler

In [22]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20],
    "min_samples_split" : [2,3,4,5]
}

In [23]:
study = optuna.create_study(direction= "maximize", sampler= optuna.samplers.GridSampler(search_space))
study.optimize(objective_function)

[I 2026-10-17 16:03:36,859] A new study created in memory with name: no-name-99d556ac-0547-4d9e-a76f-80cfc142f051
[I 2026-10-17 16:03:37,012] Trial 0 finished with value: 0.7552083333333334 and parameters: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 5}. Best is trial 0 with value: 0.7552083333333334.
[I 2026-10-17 16:03:37,163] Trial 1 finished with value: 0.7534722222222222 and parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5}. Best is trial 0 with value: 0.7552083333333334.
[I 2026-10-17 16:03:37,464] Trial 2 finished with value: 0.7586805555555555 and parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 4}. Best is trial 2 with value: 0.7586805555555555.
[I 2026-10-17 16:03:37,615] Trial 3 finished with value: 0.7569444444444443 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 5}. Best is trial 2 with value: 0.7586805555555555.
[I 2026-10-17 16:03:37,842] Trial 4 finished with value: 0.75694444444

In [24]:
study.best_params

{'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 4}

In [25]:
study.best_value

0.7760416666666666

## CMAES

In [26]:
pip install cmaes

Note: you may need to restart the kernel to use updated packages.


In [27]:
study = optuna.create_study(direction= "maximize", sampler= optuna.samplers.CmaEsSampler())   # We aim to maximize accuracy

study.optimize(objective_function, n_trials = 50)

[I 2026-10-17 16:03:49,518] A new study created in memory with name: no-name-301caf4f-522e-4abe-8901-147caac43e1f
[I 2026-10-17 16:03:49,795] Trial 0 finished with value: 0.7604166666666666 and parameters: {'n_estimators': 181, 'max_depth': 18, 'min_samples_split': 10}. Best is trial 0 with value: 0.7604166666666666.
[I 2026-10-17 16:03:50,031] Trial 1 finished with value: 0.767361111111111 and parameters: {'n_estimators': 153, 'max_depth': 12, 'min_samples_split': 8}. Best is trial 1 with value: 0.767361111111111.
[I 2026-10-17 16:03:50,210] Trial 2 finished with value: 0.7586805555555557 and parameters: {'n_estimators': 118, 'max_depth': 12, 'min_samples_split': 5}. Best is trial 1 with value: 0.767361111111111.
[I 2026-10-17 16:03:50,431] Trial 3 finished with value: 0.765625 and parameters: {'n_estimators': 146, 'max_depth': 16, 'min_samples_split': 6}. Best is trial 1 with value: 0.767361111111111.
[I 2026-10-17 16:03:50,601] Trial 4 finished with value: 0.7673611111111112 and par

In [28]:
study.best_params

{'n_estimators': 144, 'max_depth': 12, 'min_samples_split': 7}

In [29]:
study.best_value

0.7743055555555557

# Optuna Samplers

Samplers decide how Optuna proposes the next hyperparameter values in a trial. They range from exhaustive and random search to Bayesian and evolutionary methods.

## Quick Comparison

| Sampler | Type | Single / Multi-objective | Search space | Time complexity (per suggestion, approx.) | Best for |
|---|---|---|---|---|---|
| `TPESampler` | Bayesian (Tree-structured Parzen Estimator) | Both | Categorical, int, float, conditional | O(n · d · k) per suggestion | Default choice; general-purpose tuning |
| `RandomSampler` | Random | Both | All types | O(d) | Baseline; cheap trials; very high dimensions |
| `GridSampler` | Exhaustive grid | Both | Categorical, discrete | O(d) per trial; O(∏ \|vᵢ\|) total trials | Small, fully enumerable spaces |
| `BruteForceSampler` | Exhaustive enumeration | Both | Categorical only | O(d) per trial; O(∏ \|cᵢ\|) total trials | Tiny categorical spaces |
| `CmaEsSampler` | Evolutionary (CMA-ES) | Single | Continuous (int/categorical limited) | O(λ · d²) per generation, O(d³) eigendecomposition | Continuous, non-convex, moderate dimensions |
| `GPSampler` | Bayesian (Gaussian process) | Single | Continuous, int, categorical | O(n³) fitting, O(n²) per candidate | Few trials, expensive evaluations |
| `NSGAIISampler` | Evolutionary (genetic) | Multi-objective | All types | O(M · N²) non-dominated sorting per generation | Multi-objective, medium budgets |
| `NSGAIIISampler` | Evolutionary (reference points) | Multi-objective (3+ objectives) | All types | O(M · N²) plus reference-point niching | Many-objective problems |
| `QMCSampler` | Quasi-Monte Carlo (Sobol, Halton) | Both | Continuous, int | O(d) per point | Even coverage of continuous spaces |
| `PartialFixedSampler` | Wrapper | Depends on base | Depends on base | Same as base sampler | Fixing some parameters while optimizing the rest |

**Notation:** n = past trials, d = number of parameters, k = `n_ei_candidates` (default 24), λ = CMA-ES population size, N = population size, M = number of objectives, and cᵢ / vᵢ = number of categories or grid values for parameter i.

The complexity figures come from the algorithms' structure, not from Optuna's benchmarks. Real runtime depends on implementation details, the objective function's own cost, and database I/O, which often dominates for cheap objectives.

## Details

### 1. `TPESampler` (Tree-structured Parzen Estimator)

- **How it works:** Splits past trials into "good" and "bad" groups by objective value. It models each group's parameter distribution with kernel density estimators and proposes values that are more likely under the "good" model.
- **Strengths:** Robust default. Handles conditional and categorical parameters well. Low overhead per trial.
- **Limitations:** Less sample-efficient than GP-based methods on smooth, low-dimensional continuous problems. Its model is univariate by default, so it can miss parameter interactions unless `multivariate=True`.
- **Time complexity:** Roughly O(n · d · k) per suggestion, since it evaluates k candidates against KDEs built from n trials. `multivariate=True` adds cost because the joint density is modeled. Memory is O(n · d).
- **Key parameters:**
  - `n_startup_trials` (default 10): random trials before modeling starts.
  - `multivariate` (default `False`): models parameter correlations jointly.
  - `group`: groups parameters for multivariate modeling.
  - `constant_liar`: helps parallel workers avoid proposing the same point.
  - `seed`: reproducibility.
  - `consider_endpoints`, `prior_weight`, `n_ei_candidates`: fine-tuning.
- **When to use:**
  - You need a general-purpose default with little tuning.
  - The space mixes continuous, integer, and categorical parameters.
  - The space is conditional, for example, "if optimizer = Adam, tune β₁; if SGD, tune momentum."
  - You run many trials (hundreds to thousands) and the per-trial overhead matters.
  - You run distributed or parallel workers.
- **Where it works well:** ML hyperparameter tuning (learning rate, depth, regularization), pipeline or architecture search, and mixed-type configuration spaces.
- **Where it should not be used:**
  - Smooth, low-dimensional continuous problems with very few expensive trials. A GP sampler usually wins there.
  - Strongly correlated continuous parameters when you cannot enable `multivariate`. Consider CMA-ES instead.
  - Multi-objective problems with many objectives, where NSGA-II/III usually performs better.

### 2. `RandomSampler`

- **How it works:** Samples uniformly from the search space (log-uniform for log-scaled parameters).
- **Strengths:** No model fitting, so there is no overhead. Embarrassingly parallel. Surprisingly strong in high dimensions, where model-based methods struggle.
- **Limitations:** Ignores past results, so it wastes budget in low dimensions compared with model-based methods.
- **Time complexity:** O(d) per suggestion. Memory is O(1) beyond the study storage.
- **Key parameter:** `seed`.
- **When to use:**
  - As a baseline to test whether a smarter sampler actually helps.
  - When each trial is very cheap and the budget is huge.
  - When you need a quick first pass over an unfamiliar space.
  - When the objective is very noisy and a model would be misled.
- **Where it works well:** Sanity checks, high-dimensional spaces with little structure, and large parallel sweeps.
- **Where it should not be used:**
  - Expensive trials where every evaluation counts. Model-based samplers will find good values with fewer trials.
  - Tight budgets in low dimensions.

### 3. `GridSampler`

- **How it works:** Evaluates every combination in a user-defined grid. Trials stop when the grid is exhausted.
- **Strengths:** Exhaustive and deterministic. Easy to explain and reproduce.
- **Limitations:** Exponential cost in the number of parameters. Continuous parameters must be discretized, which may miss the optimum between grid points.
- **Time complexity:** The total number of trials equals the product of the grid sizes, so it grows as O(∏ |vᵢ|). Each suggestion is O(d) lookup.
- **Key parameter:** `search_space` (a dict mapping parameter names to lists of values), which must be passed explicitly.
- **When to use:**
  - The space is small and fully enumerable (for example, 3 optimizers × 4 batch sizes × 2 schedulers = 24 trials).
  - You need a fully reproducible, exhaustive record for a report or paper.
  - You are verifying the results of another sampler on a small grid.
- **Where it works well:** Small discrete ablation studies and final checks on a narrowed range.
- **Where it should not be used:**
  - Five or more parameters with reasonable grid sizes. The number of trials explodes quickly.
  - Continuous parameters where the optimum may lie between grid points.
  - Open-ended exploration of an unknown space.

### 4. `BruteForceSampler`

- **How it works:** Enumerates all categorical combinations and stops when done.
- **Strengths:** Simple and exhaustive for categorical choices.
- **Limitations:** Categorical parameters only.
- **Time complexity:** O(∏ |cᵢ|) total trials over the categorical choices.
- **When to use:** A few categorical decisions, such as choosing among preprocessing steps, model families, or feature sets, where you want every combination evaluated.
- **Where it should not be used:**
  - Any space with continuous or integer parameters.
  - Categorical spaces with many options per parameter, where the combinations explode.

### 5. `CmaEsSampler` (CMA-ES)

- **How it works:** Covariance Matrix Adaptation Evolution Strategy. It maintains a multivariate Gaussian search distribution and adapts its shape and step size based on the best candidates each generation.
- **Strengths:** Excellent on continuous, non-separable, non-convex problems. Handles correlated parameters naturally. Restarts help escape local optima.
- **Limitations:** Weak on categorical and conditional spaces. Needs the `cmaes` package. Convergence depends on population size, so it needs a reasonable trial budget.
- **Time complexity:** Each generation costs about O(λ · d²) for sampling and covariance updates, with O(d³) for eigendecomposition. `use_separable_cma` reduces the per-generation cost to roughly O(λ · d) at the expense of ignoring correlations.
- **Key parameters:**
  - `sigma0`: initial step size.
  - `n_startup_trials`: random trials before CMA-ES starts.
  - `restart_strategy` (`"ipop"` or `"bipop"`) and `inc_popsize`: restarts to escape local optima.
  - `use_separable_cma`: faster variant for high dimensions.
  - `warn_independent_sampling`.
- **When to use:**
  - Continuous parameters that interact, such as learning rate with weight decay in a tightly coupled objective.
  - Non-convex, non-separable objectives in roughly 2 to 50 dimensions.
  - You can afford a few hundred trials or more.
- **Where it works well:** Continuous controller or simulation tuning, and engineering design parameters.
- **Where it should not be used:**
  - Categorical or conditional spaces. Use TPE or GP instead.
  - Very high dimensions with a small budget. Use `use_separable_cma` or random search.
  - Multi-objective problems. Use NSGA-II/III.
  - Very few trials, where it has not yet adapted its distribution.

### 6. `GPSampler` (Gaussian Process)

- **How it works:** Fits a Gaussian process surrogate to past trials and maximizes an acquisition function (for example, Expected Improvement) to choose the next point.
- **Strengths:** Very sample-efficient. Good when each evaluation is expensive, such as a long training run or a lab experiment.
- **Limitations:** Cost grows cubically with the number of trials, so it is impractical for thousands of trials. Scales poorly beyond moderate dimensions. Model fit can be unstable with noisy objectives.
- **Time complexity:** Fitting the GP needs an O(n³) Cholesky decomposition of the n × n kernel matrix, and memory is O(n²). Scoring each candidate costs O(n²).
- **Key parameters:** `seed`, `independent_sampler`, `n_startup_trials`, `deterministic_objective`.
- **When to use:**
  - Trials are expensive (hours of GPU time or lab work) and the budget is small (tens to a few hundred trials).
  - The space is continuous or mixed, with moderate dimension.
  - The objective is smooth and close to deterministic.
- **Where it works well:** Expensive simulations, hardware or process tuning, and fine-tuning a few critical hyperparameters.
- **Where it should not be used:**
  - Thousands of trials. The O(n³) cost becomes the bottleneck.
  - Very high-dimensional spaces.
  - Noisy objectives unless you set the parameters appropriately.
  - Cheap trials, where the surrogate overhead outweighs the benefit.

### 7. `NSGAIISampler` (NSGA-II)

- **How it works:** Non-dominated Sorting Genetic Algorithm II. It evolves a population through selection, crossover, and mutation, ranking candidates by Pareto dominance and crowding distance.
- **Strengths:** Standard, reliable multi-objective method. Works for mixed parameter types. Supports constraints.
- **Limitations:** Needs a reasonable number of trials per generation. Less effective with many objectives.
- **Time complexity:** Non-dominated sorting costs about O(M · N²) per generation, where N is the population size and M the number of objectives. Crowding distance adds O(M · N log N).
- **Key parameters:**
  - `population_size` (default 50).
  - `mutation_prob` and `crossover_prob`.
  - `swapping_prob`.
  - `constraints_func`: for constrained optimization.
  - `seed`.
- **When to use:**
  - Two or three objectives that trade off, such as accuracy against latency or model size.
  - Mixed parameter types.
  - You need a Pareto front rather than a single best configuration.
- **Where it works well:** Accuracy-vs-cost tradeoffs for models, and multi-criteria engineering design.
- **Where it should not be used:**
  - Single-objective problems. A single-objective sampler will be more efficient.
  - Very small budgets (for example, under about 100 trials), where the population has not had time to evolve.
  - Many objectives (four or more). Use NSGA-III.

### 8. `NSGAIIISampler` (NSGA-III)

- **How it works:** Extends NSGA-II with reference-point-based niching, which keeps the population spread across the Pareto front in higher-dimensional objective spaces.
- **Strengths:** Better than NSGA-II when there are three or more objectives.
- **Limitations:** More setup than NSGA-II. Its advantage shrinks with only two objectives, where NSGA-II is typically sufficient.
- **Time complexity:** Similar to NSGA-II, O(M · N²) per generation for sorting, plus the cost of associating each candidate with one of H reference points (roughly O(N · H · M)). H grows quickly with M.
- **Key parameters:** `dimension` (number of objectives), `population_size`, `mutation_prob`, `crossover_prob`, `seed`.
- **When to use:** Three or more objectives, such as accuracy, latency, memory, and fairness together.
- **Where it works well:** Many-objective design problems and multi-criteria model selection.
- **Where it should not be used:**
  - Two objectives. NSGA-II is simpler and usually just as good.
  - Single-objective problems.
  - Very small budgets.

### 9. `QMCSampler` (Quasi-Monte Carlo)

- **How it works:** Generates low-discrepancy sequences (Sobol or Halton) that cover the continuous space more evenly than pure random sampling.
- **Strengths:** Better space coverage than random sampling at the same budget, with no model-fitting cost.
- **Limitations:** Continuous and integer parameters only. It does not adapt to past results, so it does not exploit good regions. Sobol sequences work best with a power-of-two number of trials.
- **Time complexity:** O(d) per point generated. Overall cost is near that of random sampling.
- **Key parameters:** `qmc_type` (`"sobol"` or `"halton"`), `scramble`, `seed`, `independent_sampler`.
- **When to use:**
  - You want a uniform initial design or a space-filling exploration phase.
  - You have a continuous space and a moderate budget, and want better coverage than random sampling.
  - You plan to refine with a model-based sampler afterward (use it as `independent_sampler` or in a warm-up phase).
- **Where it works well:** Space-filling initial designs and Monte Carlo-style sensitivity studies.
- **Where it should not be used:**
  - Categorical or conditional spaces.
  - Problems that need exploitation of past results. Use TPE, GP, or CMA-ES.

### 10. `PartialFixedSampler`

- **How it works:** Wraps another sampler and fixes a subset of parameters to specified values.
- **Strengths:** Lets you focus the search on the parameters that matter while keeping others constant.
- **Limitations:** Its behavior and efficiency depend entirely on the base sampler.
- **Time complexity:** Same as the wrapped sampler, plus negligible overhead.
- **When to use:**
  - Re-running or refining part of a search while keeping established settings fixed.
  - Staged tuning, such as optimizing the architecture first and then the regularization.
- **Where it should not be used:**
  - When you have not yet decided which parameters to fix. Fixing the wrong ones narrows the search prematurely.

## Choosing a Sampler

| Situation | Recommended sampler | Avoid |
|---|---|---|
| Not sure where to start | `TPESampler` | |
| Parameters interact, continuous | `TPESampler(multivariate=True)` or `CmaEsSampler` | Plain TPE if interactions are strong and unmodeled |
| Continuous, smooth, expensive trials, small budget | `GPSampler` | Random, grid |
| Continuous, non-convex, 2 to 50 dimensions | `CmaEsSampler` | Grid |
| Categorical or conditional space | `TPESampler` | `CmaEsSampler`, `QMCSampler` |
| Two objectives | `NSGAIISampler` | Single-objective samplers |
| Three or more objectives | `NSGAIIISampler` | `NSGAIISampler` |
| Baseline or huge cheap sweep | `RandomSampler` | `GPSampler` (overhead) |
| Small enumerable discrete space | `GridSampler` or `BruteForceSampler` | Any sampler at scale |
| Even coverage, space-filling design | `QMCSampler` | `QMCSampler` for categorical spaces |
| Fixing some parameters | `PartialFixedSampler` wrapping another sampler | |

## Usage Example

```python
import optuna
from optuna.samplers import TPESampler

sampler = TPESampler(seed=42, multivariate=True, n_startup_trials=20)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=200)
```



# study object details

In [30]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,system_attrs_cma:generation,system_attrs_cma:optimizer:0,system_attrs_cma:optimizer:1,state
0,0,0.760417,2026-10-17 16:03:49.519800,2026-10-17 16:03:49.795216,0 days 00:00:00.275416,18,10,181,NaN,NaN,NaN,COMPLETE
1,1,0.767361,2026-10-17 16:03:49.795560,2026-10-17 16:03:50.031330,0 days 00:00:00.235770,12,8,153,0.0,NaN,NaN,COMPLETE
2,2,0.758681,2026-10-17 16:03:50.031697,2026-10-17 16:03:50.210358,0 days 00:00:00.178661,12,5,118,0.0,NaN,NaN,COMPLETE
3,3,0.765625,2026-10-17 16:03:50.210693,2026-10-17 16:03:50.431066,0 days 00:00:00.220373,16,6,146,0.0,NaN,NaN,COMPLETE
4,4,0.767361,2026-10-17 16:03:50.431440,2026-10-17 16:03:50.601399,0 days 00:00:00.169959,9,8,114,0.0,NaN,NaN,COMPLETE
5,5,0.763889,2026-10-17 16:03:50.601752,2026-10-17 16:03:50.758564,0 days 00:00:00.156812,14,7,94,0.0,NaN,NaN,COMPLETE
6,6,0.758681,2026-10-17 16:03:50.758908,2026-10-17 16:03:50.941010,0 days 00:00:00.182102,10,7,121,0.0,NaN,NaN,COMPLETE
7,7,0.774306,2026-10-17 16:03:50.941347,2026-10-17 16:03:51.156564,0 days 00:00:00.215217,12,7,144,0.0,NaN,NaN,COMPLETE
8,8,0.774306,2026-10-17 16:03:51.156959,2026-10-17 16:03:51.370823,0 days 00:00:00.213864,9,8,145,1.0,800495f6060000000000008c0a636d6165732e5f636d61...,294284b014b034b0186946811894318000000000000000...,COMPLETE
9,9,0.762153,2026-10-17 16:03:51.371165,2026-10-17 16:03:51.661381,0 days 00:00:00.290216,12,7,194,1.0,NaN,NaN,COMPLETE


In [31]:
study.sampler

In [32]:
study.pruner

## Optuna Visualisation


In [33]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [34]:
#plot optimisation history
plot_optimization_history(study).show()

In [35]:
# plot paraller coordinate

plot_parallel_coordinate(study).show()

In [36]:
fig = plot_parallel_coordinate(study)
for t in fig.data:
    print(t.type)

scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter
scatter


In [37]:
fig.update_traces(
    selector=dict(type="parcoords"),
    line=dict(colorscale="Viridis"),
)
fig.show()

In [38]:
fig.update_traces(
    selector=dict(type="parcoords"),
    line=dict(cmin=0.76, cmax=0.786, colorscale="Blues"),
)
fig.show()

## slice plot



In [39]:
plot_slice(study).show()

## contour plot

In [40]:
plot_contour(study).show()

## parameter importance plot

In [41]:
plot_param_importances(study).show()

# Optimising Multiple ML models using Optuna

In [42]:
# Importing multiple models

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [43]:
def objective_function(trial):
    # give the classifier algorithms you want to test 
    classifier_name = trial.suggest_categorical("classifier", ["SVM", "GradientBoostingClassifier", "RandomForestClassifier"])

    if classifier_name == "SVM":
        c = trial.suggest_float("C", 0.1, 100, log = "True")
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly", "sigmoid"])
        gamma = trial.suggest_categorical("gamma", ["scale", "auto"])

        model = make_pipeline(
    StandardScaler(),
    SVC(kernel=kernel, C=c, gamma=gamma, max_iter=10_000, random_state=42),
)

    elif classifier_name == "RandomForestClassifier":
        n_estimators = trial.suggest_int("n_estimators", 50,300)
        max_depth = trial.suggest_int("max_depth", 3,20)
        min_samples_split = trial.suggest_int("min_samples_split", 2,10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1,10)
        bootstrap = trial.suggest_categorical("bootstrap", [True, False])

        model = RandomForestClassifier(
            n_estimators= n_estimators,
            min_samples_leaf= min_samples_leaf,
            min_samples_split= min_samples_split,
            bootstrap= bootstrap,
            max_depth= max_depth
        )

    elif classifier_name == "GradientBoostingClassifier":
        n_estimators = trial.suggest_int("n_estimators", 50,300)
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.5)
        max_depth = trial.suggest_int("max_depth", 3,20)
        min_samples_split = trial.suggest_int("min_samples_split", 2,10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1,10)

        model = GradientBoostingClassifier(
            n_estimators= n_estimators,
            learning_rate= learning_rate,
            min_samples_leaf= min_samples_leaf,
            min_samples_split= min_samples_split,
            max_depth= max_depth)

    score = cross_val_score(model, X_train, Y_train, cv= 3, scoring= "accuracy").mean()
    return score


In [44]:
import time
study = optuna.create_study(direction= "maximize", study_name= "multi_model_study", storage= "sqlite:///multi_model_study.db", load_if_exists= True)
study.optimize(objective_function, n_trials= 100)

[I 2026-10-17 16:04:03,669] A new study created in RDB with name: multi_model_study
[I 2026-10-17 16:04:03,906] Trial 0 finished with value: 0.7552083333333334 and parameters: {'classifier': 'GradientBoostingClassifier', 'n_estimators': 97, 'learning_rate': 0.3908010340819569, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.7552083333333334.
[I 2026-10-17 16:04:04,345] Trial 1 finished with value: 0.7326388888888888 and parameters: {'classifier': 'GradientBoostingClassifier', 'n_estimators': 238, 'learning_rate': 0.4484019098098168, 'max_depth': 14, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.7552083333333334.
[I 2026-10-17 16:04:04,371] Trial 2 finished with value: 0.7725694444444445 and parameters: {'classifier': 'SVM', 'C': 0.36908113655221003, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 2 with value: 0.7725694444444445.
[I 2026-10-17 16:04:04,603] Trial 3 finished with value: 0.7743055555555555 

In [45]:
study.best_value

0.7777777777777777

In [46]:
study.best_params

{'classifier': 'SVM',
 'C': 14.922780446768522,
 'kernel': 'linear',
 'gamma': 'scale'}

In [47]:
plot_param_importances(study).show()

In [48]:
plot_optimization_history(study).show()

In [49]:
plot_slice(study).show()

In [50]:
plot_parallel_coordinate(study).show()

In [51]:
study_df = study.trials_dataframe()

study_df

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.755208,2026-10-17 16:04:03.673856,2026-10-17 16:04:03.902249,0 days 00:00:00.228393,NaN,NaN,GradientBoostingClassifier,NaN,NaN,0.390801,4.0,2.0,2.0,97.0,NaN,COMPLETE
1,1,0.732639,2026-10-17 16:04:03.908929,2026-10-17 16:04:04.344130,0 days 00:00:00.435201,NaN,NaN,GradientBoostingClassifier,NaN,NaN,0.448402,14.0,2.0,4.0,238.0,NaN,COMPLETE
2,2,0.772569,2026-10-17 16:04:04.348645,2026-10-17 16:04:04.369381,0 days 00:00:00.020736,0.369081,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.774306,2026-10-17 16:04:04.375123,2026-10-17 16:04:04.601272,0 days 00:00:00.226149,NaN,False,RandomForestClassifier,NaN,NaN,NaN,19.0,9.0,10.0,167.0,NaN,COMPLETE
4,4,0.730903,2026-10-17 16:04:04.606332,2026-10-17 16:04:04.625056,0 days 00:00:00.018724,0.265718,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.703125,2026-10-17 16:04:21.567306,2026-10-17 16:04:21.614449,0 days 00:00:00.047143,89.641123,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.763889,2026-10-17 16:04:21.618336,2026-10-17 16:04:21.899268,0 days 00:00:00.280932,NaN,False,RandomForestClassifier,NaN,NaN,NaN,5.0,6.0,10.0,207.0,"{""classifier"": ""RandomForestClassifier""}",COMPLETE
97,97,0.722222,2026-10-17 16:04:21.904697,2026-10-17 16:04:21.937004,0 days 00:00:00.032307,15.811090,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.772569,2026-10-17 16:04:21.941105,2026-10-17 16:04:21.965864,0 days 00:00:00.024759,0.313323,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [52]:
study_df["params_classifier"].value_counts()

params_classifier
SVM                           54
RandomForestClassifier        36
GradientBoostingClassifier    10
Name: count, dtype: int64

In [53]:
study_df.groupby(["params_classifier"])["value"].mean()

params_classifier
GradientBoostingClassifier    0.747917
RandomForestClassifier        0.763985
SVM                           0.751511
Name: value, dtype: float64